### Load and parse part of dataset for use in LTR. 

In [1]:
import pyarrow.feather as feather
import pyarrow as pa
import glob

FEATHER_DIR = "../../baidu_ultr_dataset/parts"

# ---------------------------------------------------------
# Get complete list of columns from the first feather file
# ---------------------------------------------------------
sample_file = sorted(glob.glob(f"{FEATHER_DIR}/part-*.feather"))[0]
sample_table = feather.read_table(sample_file)

# Remove ONLY the embedding column
columns = [c for c in sample_table.column_names if c != "query_document_embedding"]

# ---------------------------------------------------------
# Load all files with the filtered column list
# ---------------------------------------------------------
paths = sorted(glob.glob(f"{FEATHER_DIR}/part-*.feather"))
# remove all with part-0 in the name
paths = [p for p in paths if "part-0" in p]

tables = []
for path in paths:
    print("Loading:", path)
    table = feather.read_table(path, columns=columns)  # all except embedding
    tables.append(table)

# Merge into a single big arrow table
ltr_table = pa.concat_tables(tables, promote=True)

print(ltr_table)

Loading: ../../baidu_ultr_dataset/parts/part-0_split-0.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-1.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-2.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-3.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-4.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-5.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-6.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-7.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-8.feather
Loading: ../../baidu_ultr_dataset/parts/part-0_split-9.feather
pyarrow.Table
query_no: int64
query_id: string
query_md5: string
url_md5: string
text_md5: string
query: list<item: int64>
  child 0, item: int64
title: list<item: int64>
  child 0, item: int64
abstract: list<item: int64>
  child 0, item: int64
position: int64
media_type: int64
displayed_time: double
serp_height: int64
slipoff_count_after_click: int64
click: int64
bm2

/opt/homebrew/Caskroom/miniforge/base/envs/two-tower-confounding-2/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3579: FutureWarning: promote has been superseded by promote_options='default'.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
import numpy as np
import pyarrow as pa
from typing import Tuple, List, Dict, Any

def extract_arrays(ltr_table: pa.Table) -> Dict[str, np.ndarray]:
    """Extract primary numpy arrays from arrow table."""
    query_no   = ltr_table["query_no"].to_numpy()
    positions  = ltr_table["position"].to_numpy() - 1  # zero-based
    clicks     = ltr_table["click"].to_numpy()
    query_md5  = ltr_table["query_md5"].to_numpy()
    return {"query_no": query_no, "positions": positions, "clicks": clicks, "query_md5": query_md5}

def session_boundaries(query_no: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    unique_sessions, start_idx = np.unique(query_no, return_index=True)
    end_idx = np.append(start_idx[1:], len(query_no))
    return unique_sessions, start_idx, end_idx

def build_ragged_lists(positions: np.ndarray, clicks: np.ndarray, start_idx: np.ndarray, end_idx: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    all_positions = [positions[s:e] for s, e in zip(start_idx, end_idx)]
    all_clicks    = [clicks[s:e]    for s, e in zip(start_idx, end_idx)]
    return all_positions, all_clicks

def compute_target_lengths(all_positions: List[np.ndarray]) -> np.ndarray:
    obs_lens = np.array([arr.size for arr in all_positions], dtype=np.int32)
    max_positions = np.array([int(arr.max()) if arr.size>0 else -1 for arr in all_positions], dtype=np.int32)
    target_lengths = np.where(obs_lens>0, max_positions + 1, 0)
    return target_lengths

def flatten_and_scatter(all_positions: List[np.ndarray], all_clicks: List[np.ndarray], target_lengths: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    row_offsets = np.concatenate([[0], np.cumsum(target_lengths)[:-1]]).astype(np.int64)
    total_cells = int(row_offsets[-1] + target_lengths[-1]) if target_lengths.size else 0
    flat_positions = np.full(total_cells, -1, dtype=np.int32)
    flat_clicks = np.zeros(total_cells, dtype=np.int8)
    for i, (pos_arr, clk_arr) in enumerate(zip(all_positions, all_clicks)):
        L = pos_arr.size
        offset = row_offsets[i]
        if L == target_lengths[i]:
            flat_positions[offset:offset+L] = pos_arr
            flat_clicks[offset:offset+L] = clk_arr
        else:
            max_p = target_lengths[i]
            tmp_pos = np.full(max_p, -1, dtype=np.int32)
            tmp_clk = np.zeros(max_p, dtype=np.int8)
            tmp_pos[pos_arr] = pos_arr
            tmp_clk[pos_arr] = clk_arr
            flat_positions[offset:offset+max_p] = tmp_pos
            flat_clicks[offset:offset+max_p] = tmp_clk
    num_sessions = len(target_lengths)
    return flat_positions, flat_clicks, row_offsets, target_lengths

def reshape_to_padded(flat_positions: np.ndarray, flat_clicks: np.ndarray, row_offsets: np.ndarray, target_lengths: np.ndarray, num_sessions: int, max_len: int) -> Tuple[np.ndarray, np.ndarray]:
    padded_positions = np.full((num_sessions, max_len), -1, dtype=np.int32)
    padded_clicks    = np.zeros((num_sessions, max_len), dtype=np.int8)
    for i in range(num_sessions):
        L = target_lengths[i]
        if L == 0:
            continue
        L = min(target_lengths[i], max_len)
        off = row_offsets[i]
        padded_positions[i, :L] = flat_positions[off:off+L]
        padded_clicks[i, :L]    = flat_clicks[off:off+L]
    return padded_positions, padded_clicks

def build_mask(padded_positions: np.ndarray) -> np.ndarray:
    return (padded_positions >= 0).astype(np.int8)

def sessions_per_query_from_md5(query_md5: np.ndarray, start_idx: np.ndarray, end_idx: np.ndarray) -> np.ndarray:
    if len(start_idx) == 0:
        return np.zeros(0, dtype=np.int32)
    session_query_md5 = np.array([query_md5[s:e][0] for s, e in zip(start_idx, end_idx)])
    unique_queries, query_counts = np.unique(session_query_md5, return_counts=True)
    query_to_count = dict(zip(unique_queries, query_counts))
    sessions_per_query = np.array([query_to_count[q] for q in session_query_md5], dtype=np.int32)
    return sessions_per_query

def sessions_per_doc_pos_matrix(padded_positions: np.ndarray, mask: np.ndarray) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    sessions_per_doc_pos = np.zeros((num_sessions, max_len, max_len), dtype=np.int32)
    for i in range(num_sessions):
        pos = padded_positions[i]
        L = np.sum(mask[i])
        sessions_per_doc_pos[i, np.arange(L), pos[:L]] = 1
    return sessions_per_doc_pos

def extract_lp_features(ltr_table: pa.Table, padded_positions: np.ndarray, lp_feature_names: List[str]) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    lp_features_arr = np.stack([ltr_table[name].to_numpy() for name in lp_feature_names], axis=1)  # (N_total, n_features)
    mask_valid = padded_positions.ravel() >= 0
    n_features = len(lp_feature_names)
    lp_query_doc_features = np.zeros((num_sessions, max_len, n_features), dtype=np.float32)
    lp_query_doc_features.reshape(-1, n_features)[mask_valid] = lp_features_arr
    return lp_query_doc_features

def extract_query_doc_features(ltr_table: pa.Table, padded_positions: np.ndarray, feature_names: List[str]) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    query_features_arr = np.stack([ltr_table[name].to_numpy().astype(np.float32) for name in feature_names], axis=1)  # (N_total, n_features)
    mask_valid = padded_positions.ravel() >= 0
    n_features = len(feature_names)
    query_doc_features = np.zeros((num_sessions, max_len, n_features), dtype=np.float32)
    query_doc_features.reshape(-1, n_features)[mask_valid] = query_features_arr
    return query_doc_features

def build_all(ltr_table: pa.Table) -> Dict[str, Any]:
    MAX_LEN = 20
    # 1. Extract arrays
    positions_raw = ltr_table["position"].to_numpy() - 1
    valid_mask = positions_raw < MAX_LEN
    ltr_table = ltr_table.filter(valid_mask)
    arrs = extract_arrays(ltr_table)
    
    # 2. Session boundaries
    unique_sessions, start_idx, end_idx = session_boundaries(arrs["query_no"])
    # 3. Ragged lists
    all_positions, all_clicks = build_ragged_lists(arrs["positions"], arrs["clicks"], start_idx, end_idx)
    # 4. Target lengths
    target_lengths = compute_target_lengths(all_positions)
    num_sessions = len(all_positions)
    # 5. Flatten and scatter
    flat_positions, flat_clicks, row_offsets, target_lengths = flatten_and_scatter(all_positions, all_clicks, target_lengths)
    # 6. Reshape
    padded_positions, padded_clicks = reshape_to_padded(flat_positions, flat_clicks, row_offsets, target_lengths, num_sessions, MAX_LEN)
    # 7. Mask
    mask = build_mask(padded_positions)
    # 8. sessions_per_query
    sessions_per_query = sessions_per_query_from_md5(arrs["query_md5"], start_idx, end_idx)
    # 9. sessions_per_doc_pos
    sessions_per_doc_pos = sessions_per_doc_pos_matrix(padded_positions, mask)
    # 10. features
    lp_feature_names = ["position", "media_type", "displayed_time", "serp_height", "slipoff_count_after_click"]
    lp_query_doc_features = extract_lp_features(ltr_table, padded_positions, lp_feature_names)
    feature_names = [
        "bm25", "bm25_title", "bm25_abstract",
        "tf_idf", "tf", "idf",
        "ql_jelinek_mercer_short", "ql_jelinek_mercer_long", "ql_dirichlet",
        "query_length", "document_length", "title_length", "abstract_length"
    ]
    query_doc_features = extract_query_doc_features(ltr_table, padded_positions, feature_names)
    # 11. placeholders
    query_doc_ids = np.zeros((num_sessions, MAX_LEN), dtype=np.int64)
    labels        = padded_clicks.copy()
    n             = np.ones(num_sessions, dtype=np.int32)
    # queries = np.array([ltr_table["query"][s:e][0] for s, e in zip(start_idx, end_idx)], dtype=object)
    query_no = np.array([ltr_table["query_no"][s:e][0] for s, e in zip(start_idx, end_idx)], dtype=np.int64)
    return {
        "padded_positions": padded_positions,
        "padded_clicks": padded_clicks,
        "mask": mask,
        "sessions_per_query": sessions_per_query,
        "sessions_per_doc_pos": sessions_per_doc_pos,
        "lp_query_doc_features": lp_query_doc_features,
        "query_doc_features": query_doc_features,
        "query_doc_ids": query_doc_ids,
        "labels": labels,
        "n": n,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "unique_sessions": unique_sessions,
        "queries": query_no,
    }

# Runner: call build_all and print shapes
result = build_all(ltr_table)
padded_positions = result["padded_positions"]
padded_clicks = result["padded_clicks"]
mask = result["mask"]
sessions_per_query = result["sessions_per_query"]
sessions_per_doc_pos = result["sessions_per_doc_pos"]
lp_query_doc_features = result["lp_query_doc_features"]
query_doc_features = result["query_doc_features"]
query_doc_ids = result["query_doc_ids"]
labels = result["labels"]
n = result["n"]
start_idx = result["start_idx"]
end_idx = result["end_idx"]
unique_sessions = result["unique_sessions"]
queries = result["queries"]

np.savez_compressed(
    "train_Baidu_ULTRA.npz",
    padded_positions=padded_positions,
    padded_clicks=padded_clicks,
    mask=mask,
    sessions_per_query=sessions_per_query,
    sessions_per_doc_pos=sessions_per_doc_pos,
    query_doc_features=query_doc_features,
    lp_query_doc_features=lp_query_doc_features,
    query_doc_ids=query_doc_ids,
    n=n,
    queries=queries,
    labels=labels
)

In [9]:
import numpy as np
import pyarrow as pa
from typing import Tuple, List, Dict, Any

def extract_arrays(ltr_table: pa.Table) -> Dict[str, np.ndarray]:
    """Extract primary numpy arrays from arrow table."""
    query_no   = ltr_table["query_no"].to_numpy()
    positions  = ltr_table["position"].to_numpy() - 1  # zero-based
    clicks     = ltr_table["click"].to_numpy()
    query_md5  = ltr_table["query_md5"].to_numpy()
    return {"query_no": query_no, "positions": positions, "clicks": clicks, "query_md5": query_md5}

def session_boundaries(query_no: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    unique_sessions, start_idx = np.unique(query_no, return_index=True)
    end_idx = np.append(start_idx[1:], len(query_no))
    return unique_sessions, start_idx, end_idx

def build_ragged_lists(positions: np.ndarray, clicks: np.ndarray, start_idx: np.ndarray, end_idx: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    all_positions = [positions[s:e] for s, e in zip(start_idx, end_idx)]
    all_clicks    = [clicks[s:e]    for s, e in zip(start_idx, end_idx)]
    return all_positions, all_clicks

def compute_target_lengths(all_positions: List[np.ndarray]) -> np.ndarray:
    obs_lens = np.array([arr.size for arr in all_positions], dtype=np.int32)
    max_positions = np.array([int(arr.max()) if arr.size>0 else -1 for arr in all_positions], dtype=np.int32)
    target_lengths = np.where(obs_lens>0, max_positions + 1, 0)
    return target_lengths

def flatten_and_scatter(all_positions: List[np.ndarray], all_clicks: List[np.ndarray], target_lengths: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    row_offsets = np.concatenate([[0], np.cumsum(target_lengths)[:-1]]).astype(np.int64)
    total_cells = int(row_offsets[-1] + target_lengths[-1]) if target_lengths.size else 0
    flat_positions = np.full(total_cells, -1, dtype=np.int32)
    flat_clicks = np.zeros(total_cells, dtype=np.int8)
    for i, (pos_arr, clk_arr) in enumerate(zip(all_positions, all_clicks)):
        L = pos_arr.size
        offset = row_offsets[i]
        if L == target_lengths[i]:
            flat_positions[offset:offset+L] = pos_arr
            flat_clicks[offset:offset+L] = clk_arr
        else:
            max_p = target_lengths[i]
            tmp_pos = np.full(max_p, -1, dtype=np.int32)
            tmp_clk = np.zeros(max_p, dtype=np.int8)
            tmp_pos[pos_arr] = pos_arr
            tmp_clk[pos_arr] = clk_arr
            flat_positions[offset:offset+max_p] = tmp_pos
            flat_clicks[offset:offset+max_p] = tmp_clk
    num_sessions = len(target_lengths)
    return flat_positions, flat_clicks, row_offsets, target_lengths

def reshape_to_padded(flat_positions: np.ndarray, flat_clicks: np.ndarray, row_offsets: np.ndarray, target_lengths: np.ndarray, num_sessions: int, max_len: int) -> Tuple[np.ndarray, np.ndarray]:
    padded_positions = np.full((num_sessions, max_len), -1, dtype=np.int32)
    padded_clicks    = np.zeros((num_sessions, max_len), dtype=np.int8)
    for i in range(num_sessions):
        L = target_lengths[i]
        if L == 0:
            continue
        L = min(target_lengths[i], max_len)
        off = row_offsets[i]
        padded_positions[i, :L] = flat_positions[off:off+L]
        padded_clicks[i, :L]    = flat_clicks[off:off+L]
    return padded_positions, padded_clicks

def build_mask(padded_positions: np.ndarray) -> np.ndarray:
    return (padded_positions >= 0).astype(np.int8)

def sessions_per_query_from_md5(query_md5: np.ndarray, start_idx: np.ndarray, end_idx: np.ndarray) -> np.ndarray:
    if len(start_idx) == 0:
        return np.zeros(0, dtype=np.int32)
    session_query_md5 = np.array([query_md5[s:e][0] for s, e in zip(start_idx, end_idx)])
    unique_queries, query_counts = np.unique(session_query_md5, return_counts=True)
    query_to_count = dict(zip(unique_queries, query_counts))
    sessions_per_query = np.array([query_to_count[q] for q in session_query_md5], dtype=np.int32)
    return sessions_per_query

def sessions_per_doc_pos_matrix(padded_positions: np.ndarray, mask: np.ndarray) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    sessions_per_doc_pos = np.zeros((num_sessions, max_len, max_len), dtype=np.int32)
    for i in range(num_sessions):
        pos = padded_positions[i]
        L = np.sum(mask[i])
        sessions_per_doc_pos[i, np.arange(L), pos[:L]] = 1
    return sessions_per_doc_pos

def extract_lp_features(ltr_table: pa.Table, padded_positions: np.ndarray, lp_feature_names: List[str]) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    lp_features_arr = np.stack([ltr_table[name].to_numpy() for name in lp_feature_names], axis=1)  # (N_total, n_features)
    mask_valid = padded_positions.ravel() >= 0
    n_features = len(lp_feature_names)
    lp_query_doc_features = np.zeros((num_sessions, max_len, n_features), dtype=np.float32)
    lp_query_doc_features.reshape(-1, n_features)[mask_valid] = lp_features_arr
    return lp_query_doc_features

def extract_query_doc_features(ltr_table: pa.Table, padded_positions: np.ndarray, feature_names: List[str]) -> np.ndarray:
    num_sessions, max_len = padded_positions.shape
    query_features_arr = np.stack([ltr_table[name].to_numpy().astype(np.float32) for name in feature_names], axis=1)  # (N_total, n_features)
    mask_valid = padded_positions.ravel() >= 0
    n_features = len(feature_names)
    query_doc_features = np.zeros((num_sessions, max_len, n_features), dtype=np.float32)
    query_doc_features.reshape(-1, n_features)[mask_valid] = query_features_arr
    return query_doc_features

def build_all(ltr_table: pa.Table) -> Dict[str, Any]:
    MAX_LEN = 10
    # 1. Extract arrays
    positions_raw = ltr_table["position"].to_numpy() - 1
    valid_mask = positions_raw < MAX_LEN
    ltr_table = ltr_table.filter(valid_mask)
    arrs = extract_arrays(ltr_table)
    
    # 2. Session boundaries
    unique_sessions, start_idx, end_idx = session_boundaries(arrs["query_no"])
    # 3. Ragged lists
    all_positions, all_clicks = build_ragged_lists(arrs["positions"], arrs["clicks"], start_idx, end_idx)
    # 4. Target lengths
    target_lengths = compute_target_lengths(all_positions)
    num_sessions = len(all_positions)
    # 5. Flatten and scatter
    flat_positions, flat_clicks, row_offsets, target_lengths = flatten_and_scatter(all_positions, all_clicks, target_lengths)
    # 6. Reshape
    padded_positions, padded_clicks = reshape_to_padded(flat_positions, flat_clicks, row_offsets, target_lengths, num_sessions, MAX_LEN)
    # 7. Mask
    mask = build_mask(padded_positions)
    # 8. sessions_per_query
    sessions_per_query = sessions_per_query_from_md5(arrs["query_md5"], start_idx, end_idx)
    # 9. sessions_per_doc_pos
    sessions_per_doc_pos = sessions_per_doc_pos_matrix(padded_positions, mask)
    # 10. features
    lp_feature_names = ["position", "media_type", "displayed_time", "serp_height", "slipoff_count_after_click"]
    lp_query_doc_features = extract_lp_features(ltr_table, padded_positions, lp_feature_names)
    feature_names = [
        "bm25", "bm25_title", "bm25_abstract",
        "tf_idf", "tf", "idf",
        "ql_jelinek_mercer_short", "ql_jelinek_mercer_long", "ql_dirichlet",
        "query_length", "document_length", "title_length", "abstract_length"
    ]
    query_doc_features = extract_query_doc_features(ltr_table, padded_positions, feature_names)
    # 11. placeholders
    query_doc_ids = np.zeros((num_sessions, MAX_LEN), dtype=np.int64)
    labels        = padded_clicks.copy()
    n             = np.ones(num_sessions, dtype=np.int32)
    # queries = np.array([ltr_table["query"][s:e][0] for s, e in zip(start_idx, end_idx)], dtype=object)
    query_no = np.array([ltr_table["query_no"][s:e][0] for s, e in zip(start_idx, end_idx)], dtype=np.int64)
    return {
        "padded_positions": padded_positions,
        "padded_clicks": padded_clicks,
        "mask": mask,
        "sessions_per_query": sessions_per_query,
        "sessions_per_doc_pos": sessions_per_doc_pos,
        "lp_query_doc_features": lp_query_doc_features,
        "query_doc_features": query_doc_features,
        "query_doc_ids": query_doc_ids,
        "labels": labels,
        "n": n,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "unique_sessions": unique_sessions,
        "queries": query_no,
    }

# Runner: call build_all and print shapes
result = build_all(ltr_table)
padded_positions = result["padded_positions"]
padded_clicks = result["padded_clicks"]
mask = result["mask"]
sessions_per_query = result["sessions_per_query"]
sessions_per_doc_pos = result["sessions_per_doc_pos"]
lp_query_doc_features = result["lp_query_doc_features"]
query_doc_features = result["query_doc_features"]
query_doc_ids = result["query_doc_ids"]
labels = result["labels"]
n = result["n"]
start_idx = result["start_idx"]
end_idx = result["end_idx"]
unique_sessions = result["unique_sessions"]
queries = result["queries"]

np.savez_compressed(
    "train_Baidu_ULTRA_very_short.npz",
    padded_positions=padded_positions,
    padded_clicks=padded_clicks,
    mask=mask,
    sessions_per_query=sessions_per_query,
    sessions_per_doc_pos=sessions_per_doc_pos,
    query_doc_features=query_doc_features,
    lp_query_doc_features=lp_query_doc_features,
    query_doc_ids=query_doc_ids,
    n=n,
    queries=queries,
    labels=labels
)